# பாடம் 05 - முகவர் RAG


## அமைப்பு

இந்த நோட்டுபுத்தகம் Microsoft Agent Framework பயன்படுத்தி Agentic RAG (Retrieval-Augmented Generation) மாதிரியை விளக்குகிறது.

**முன்னேற்பாடு:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — உங்கள் Azure AI Search சேவை முனையம்
- `AZURE_SEARCH_API_KEY` — உங்கள் Azure AI Search API விசை
- சூழல் மாறிகளின் மூலம் Azure OpenAI இடம்பெறும்
- Azure CLI அங்கீகாரம் பெற்றது (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG என்ன?

சாதாரண RAG ஒரு நிலையான குழாய்த் சுரங்கத்தை பின்பற்றுகிறது: ஆவணங்களை மீட்டெடுத்து, பிறகு பதிலை உருவாக்குகிறது. **Agentic RAG** தகவலை எப்போது மற்றும் எவ்வாறு மீட்டெடுக்க வேண்டும் என்பதை முகவருக்கு தன்னாட்சி வழங்குவதன் மூலம் இது மேலும்முன்நிலை செல்வதைக் குறிக்கிறது.

Agentic RAG உடன், முகவர் முடியும்:
- கேள்விக்கு பதிலிடுவதற்கு முன்னர் மீட்டெடுப்பு தேவைப்படுமா என்று **தீர்மானிக்க** 
- எந்த தரவுத்தளம் அல்லது கருவி விசாரணை செய்யப்பட வேண்டும் என்று **தேர்ந்தெடுக்க** 
- மீட்டெடுக்கப்பட்ட முடிவுகளை **மதிப்பாய்வு செய்து** தேவையானால் தொடர்ந்த மீட்டெடுப்புகளைச் செய்ய
- பல மீட்டெடுப்பு படிகளிலிருந்து தகவலை ஒருங்கிணைத்து தெளிவான பதிலாக **இணைக்க**

இதனால் முகவர் நிலையான மீட்டெடு-பின்-உருவாக்க குழாய்த் சுரங்கத்தைவிட அதிக தகுதியும் துல்லியமும் பெறுகிறார்.


## தேடல் கருவியை உருவாக்குதல்

Agentic RAG இல், வெளிப்புற தரவுத் स्रोतங்கள் முகவர்கள் வேண்டுகோளுக்கு ஏற்ப அணுகக்கூடிய **கருவிகள்** என ஒரு பூட்டியுடன் கொடுக்கப்படுகின்றன. இதனால் முகவர் எதிர்பாராமல் செய்யகூடிய மற்றொரு நடவடிக்கையாக தேடல் செயல்பாட்டைப் பார்கின்றார், கட்டாயமான படிமுறை அல்லாமல்.

அடிப்படையில் ஒரு பயண அறிவுத்தளத்தை வரையறுத்து, அதை முகவர் இடம் தொடர்பான தகவல்களை தேட அழைக்கக்கூடிய கருவியாக வெளிப்படுத்துகிறோம்.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG முகவரியை உருவாக்குதல்

இப்போது நாம் **எப்போதும் பதிலளிப்பதற்கு முன்பு தகவலை மீட்டெடுக்க** அறிவுறுத்தப்பட்ட முகவரியை உருவாக்குகிறோம். முகவர் தனது பதில்களை தனது சொந்த பயிற்சி தரவிற்கு பதிலாக அறிவு அடிப்படையில் நிலையானதாக மாற்ற `search_travel_knowledge` கருவியை பயன்படுத்துகிறது.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## முறைமையான மீட்பு — மேக்கர்-செக்கர் வடிவம்

Agentic RAG இன் முக்கியமான நன்மை **முறைமையான மீட்பு** ஆகும். முகவர் தன் ஆரம்ப கண்டுபிடிப்புகளை உறுதிப்படுத்த, நுட்பப்படுத்த அல்லது விரிவுபடுத்த பல சுற்று தேடல்களை செய்ய முடியும் — இது "மேக்கர்-செக்கர்" பணிநிரல் போன்றதாகும்:

1. **மேக்கர் படி**: முகவர் தொடக்க தகவல்களை மீட்டெடுத்து பதிலை வரைந்தெடுக்கும்.
2. **செக்கர் படி**: முகவர் விவரங்களை உறுதிப்படுத்த அல்லது இடைவெளிகளை நிரப்ப கூடுதல் மீட்புகளை செய்யும்.

கீழே, முகவரிடம் பல வழிகள் ஒப்பிட வேண்டிய கேள்வி கேட்கப்படுகிறது, இதனால் அது பல முறை தேட வேண்டியதாகும்.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Frameworkஐ பயன்படுத்தி **Agentic RAG** அமைப்பை எவ்வாறு உருவாக்குவது என்பதை கற்று கொண்டீர்கள்:

- **Agentic RAG** агентக்கள் தன்னிச்சையாக தகவலை எப்போது மீட்டெடுக்க வேண்டுமென்று தீர்மானிக்க அனுமதிக்கிறது, இதனால் மீட்டெடுப்பு நிலையானது அல்ல, மாறும் வகையில் இருக்கும்.
- **கருவிகள் தரவுத் தளங்களாக**: வெளியுறைய அறிவுக் கொள்கைகள் (Azure AI Search போன்றவை) агентக் கூரை வரியாக உபகரணங்களாக மடியப்பட்டுள்ளன.
- **மடங்கான மீட்டெடுப்பு**: உருவாக்கி-சரிபார்ப்பவர் வடிவமைப்பு агентக்கு பலமடங்கு மீட்டெடுப்பு சுற்றங்களை செய்ய உதவுகிறது — தேடுதல், சரிபார்ப்பு, மற்றும் புத்திசாலித்தனமாக மின்னல் செய்வது — இறுதி பதிலை வழங்குவதற்குமுன்பு.

உருவாக்கத்தில், பெரிய அளவிலான பயண ஆவண மீட்டெடுப்பு கையாள, நினைவகத்தில் உள்ள `TRAVEL_KNOWLEDGE_BASE` ஐ ஒத்துப்போகும் ஒரு உண்மையான Azure AI Search குறியீட்டோடு மாற்றுவீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**குறிப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்புச் சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாம் துல்லியத்திற்காக முயற்சித்தாலும், சுயமாக இயங்கும் மொழிபெயர்ப்பில் தவறுகள் அல்லது பிழைகள் இருக்க வாய்ப்பு உள்ளது. அசல் ஆவணம் அதன் சொந்த மொழியில் அதிகாரப்பூர்வமாகக் கருதப்பட வேண்டும். முக்கிய தகவல்களுக்கு, தரமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயனிலிருந்து ஏற்படும் எந்த தவறுகளைப் பொறுப்பேற்கவில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
